# Customer 360 — Project Architecture
### Built on Databricks | Medallion Architecture (Bronze → Silver → Gold)
---
**What this project does:**
A single, unified and complete view of every bank customer —
combining identity, accounts, loans, cards, transactions,
digital activity, interactions and notifications into one profile.

In [0]:
html_architecture = """
<style>
  body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }
  .arch-container { max-width: 1100px; margin: 0 auto; padding: 20px; }
  .arch-title { text-align: center; font-size: 22px; font-weight: 600;
                color: #2E8B57; margin-bottom: 30px; }
  .layer-row { display: flex; align-items: stretch;
               margin-bottom: 6px; gap: 8px; }
  .layer-label { width: 110px; display: flex; align-items: center;
                 justify-content: center; font-weight: 600;
                 font-size: 13px; border-radius: 8px; padding: 10px 6px;
                 text-align: center; flex-shrink: 0; }
  .layer-content { flex: 1; display: flex; gap: 6px;
                   flex-wrap: wrap; align-items: center; }
  .table-box { padding: 6px 10px; border-radius: 6px;
               font-size: 11px; font-weight: 500; text-align: center;
               min-width: 100px; border: 1px solid; }
  .arrow-row { display: flex; justify-content: center;
               align-items: center; color: #2E8B57; font-size: 20px;
               margin: 4px 0; }
  /* SOURCE */
  .lbl-src { background:#2A2A3A; color:#AAB; border:1.5px solid #445; }
  .box-src { background:#1E1E2E; color:#99AACC;
             border-color:#334; font-size:10px; }
  /* BRONZE */
  .lbl-brz { background:#3D2B00; color:#FFB347; border:1.5px solid #8B5E00; }
  .box-brz { background:#2B1E00; color:#FFB347; border-color:#8B5E00; }
  /* SILVER */
  .lbl-slv { background:#1A2A3A; color:#7EC8E3; border:1.5px solid #2E6A8A; }
  .box-slv { background:#0F1E2E; color:#7EC8E3; border-color:#2E6A8A; }
  /* GOLD */
  .lbl-gld { background:#2A2A00; color:#FFD700; border:1.5px solid #8B8B00; }
  .box-gld { background:#1A1A00; color:#FFD700; border-color:#8B8B00; }
  /* DASHBOARD */
  .lbl-dsh { background:#1A2E1A; color:#90EE90; border:1.5px solid #2E6B2E; }
  .box-dsh { background:#0F1E0F; color:#90EE90; border-color:#2E6B2E; }
  /* MAP */
  .box-map { background:#2A1A2A; color:#CC99FF; border-color:#6633AA;
             font-size:10px; }
  .section-note { font-size:10px; color:#333; margin-top:3px;
                  padding-left:118px; }
  .stat-bar { display:flex; justify-content:center; gap:40px;
              margin:20px 0; padding:14px; background:#1A1A2A;
              border-radius:10px; border:1px solid #333; }
  .stat-item { text-align:center; }
  .stat-num { font-size:24px; font-weight:700; color:#FFD700; }
  .stat-lbl { font-size:11px; color:#888; margin-top:3px; }
</style>

<div class="arch-container">
  <div class="arch-title">Customer 360 — Databricks Medallion Architecture</div>

  <!-- STATS BAR -->
  <div class="stat-bar">
    <div class="stat-item">
      <div class="stat-num">11</div>
      <div class="stat-lbl">Source CSV files</div>
    </div>
    <div class="stat-item">
      <div class="stat-num">1.17M</div>
      <div class="stat-lbl">Total rows ingested</div>
    </div>
    <div class="stat-item">
      <div class="stat-num">10</div>
      <div class="stat-lbl">Silver tables</div>
    </div>
    <div class="stat-item">
      <div class="stat-num">6</div>
      <div class="stat-lbl">Gold tables</div>
    </div>
    <div class="stat-item">
      <div class="stat-num">50K</div>
      <div class="stat-lbl">Customer profiles</div>
    </div>
    <div class="stat-item">
      <div class="stat-num">11</div>
      <div class="stat-lbl">Dashboard panels</div>
    </div>
  </div>

  <!-- SOURCE LAYER -->
  <div class="layer-row">
    <div class="layer-label lbl-src">Source<br>CSV Files</div>
    <div class="layer-content">
      <div class="table-box box-src">customers.csv<br>50K rows</div>
      <div class="table-box box-src">accounts.csv<br>75K rows</div>
      <div class="table-box box-src">cards.csv<br>100K rows</div>
      <div class="table-box box-src">loans.csv<br>30K rows</div>
      <div class="table-box box-src">branches.csv<br>500 rows</div>
      <div class="table-box box-src">merchants.csv<br>5K rows</div>
      <div class="table-box box-src">transactions.csv<br>500K rows</div>
      <div class="table-box box-src">digital_activity.csv<br>200K rows</div>
      <div class="table-box box-src">loan_payments.csv<br>60K rows</div>
      <div class="table-box box-src">cust_interactions.csv<br>50K rows</div>
      <div class="table-box box-src">notifications.csv<br>100K rows</div>
    </div>
  </div>
  <div class="section-note">
    Stored in Unity Catalog Volume:
    /Volumes/customer_360/source_data/raw/
  </div>

  <div class="arrow-row">↓ read_files() + schema enforcement + _metadata audit columns</div>

  <!-- BRONZE LAYER -->
  <div class="layer-row">
    <div class="layer-label lbl-brz">Bronze<br>Raw Delta</div>
    <div class="layer-content">
      <div class="table-box box-brz">bronze_customers</div>
      <div class="table-box box-brz">bronze_accounts</div>
      <div class="table-box box-brz">bronze_cards</div>
      <div class="table-box box-brz">bronze_loans</div>
      <div class="table-box box-brz">bronze_branches</div>
      <div class="table-box box-brz">bronze_merchants</div>
      <div class="table-box box-brz">bronze_transactions</div>
      <div class="table-box box-brz">bronze_digital_activity</div>
      <div class="table-box box-brz">bronze_loan_payment</div>
      <div class="table-box box-brz">bronze_customer_interactions</div>
      <div class="table-box box-brz">bronze_notifications</div>
    </div>
  </div>
  <div class="section-note">
    Exact copy of source · No transforms · Audit columns:
    ingestion_time, source_file, _rescued_data
  </div>

  <div class="arrow-row">
    ↓ ID reconciliation + dedup + null handling +
    type casting + business rules + derived columns
  </div>

  <!-- SILVER LAYER -->
  <div class="layer-row">
    <div class="layer-label lbl-slv">Silver<br>Clean</div>
    <div class="layer-content">
      <div class="table-box box-slv">silver_customers<br>full_name · credit_band</div>
      <div class="table-box box-slv">silver_accounts<br>balance_tier · age_days</div>
      <div class="table-box box-slv">silver_cards<br>card_status · expiry</div>
      <div class="table-box box-slv">silver_loans<br>loan_size · risk_flag</div>
      <div class="table-box box-slv">silver_branches<br>branch_city</div>
      <div class="table-box box-slv">silver_merchants<br>merchant_city</div>
      <div class="table-box box-slv">silver_transactions<br>spending_amount · year_month</div>
      <div class="table-box box-slv">silver_digital_activity<br>is_mobile · engagement flags</div>
      <div class="table-box box-slv">silver_loan_payments<br>risk_tier · payment_gap</div>
      <div class="table-box box-slv">silver_customer_interactions<br>is_branch · is_complaint</div>
      <div class="table-box box-slv">silver_notifications<br>engagement_score</div>
    </div>
  </div>
  <div class="layer-row" style="margin-top:4px">
    <div class="layer-label" style="background:#1A1A2A;color:#CC99FF;
         border:1.5px solid #6633AA;font-size:11px;">ID Mapping<br>Tables</div>
    <div class="layer-content">
      <div class="table-box box-map">map_customer_ids<br>CUST* → CUS*</div>
      <div class="table-box box-map">map_account_ids<br>ACC* → ACC*</div>
      <div class="table-box box-map">map_card_ids<br>CARD* → CRD*</div>
      <div class="table-box box-map">map_loan_ids<br>LOAN* → LON*</div>
      <div class="table-box box-map">map_merchant_ids<br>MER* → MER*</div>
      <div class="table-box box-map">map_branch_ids<br>BR* → BRN*</div>
    </div>
  </div>
  <div class="section-note">
    Silver = trusted, typed, deduplicated data ·
    ID reconciliation links all 11 datasets to a single customer_id
  </div>

  <div class="arrow-row">
    ↓ 8-way aggregation JOIN ·
    engagement score formula · churn risk formula · customer tags
  </div>

  <!-- GOLD LAYER -->
  <div class="layer-row">
    <div class="layer-label lbl-gld">Gold<br>Business</div>
    <div class="layer-content">
      <div class="table-box box-gld">customer_360<br>1 row per customer<br>50K rows</div>
      <div class="table-box box-gld">monthly_spending<br>spend trend<br>per month</div>
      <div class="table-box box-gld">product_holdings<br>balance per<br>product type</div>
      <div class="table-box box-gld">channel_usage<br>mobile vs<br>desktop %</div>
      <div class="table-box box-gld">churn_risk_monthly<br>risk score<br>over time</div>
      <div class="table-box box-gld">activity_feed<br>merged event<br>timeline</div>
    </div>
  </div>
  <div class="section-note">
    Gold = pre-aggregated, dashboard-ready ·
    Engagement score (0-100) · Churn risk score (0-100) ·
    Tags: High Value · Digital-first · Low churn risk
  </div>

  <div class="arrow-row">↓ Databricks AI/BI Dashboard · Field filter by customer_id</div>

  <!-- DASHBOARD LAYER -->
  <div class="layer-row">
    <div class="layer-label lbl-dsh">Dashboard<br>Customer<br>360</div>
    <div class="layer-content">
      <div class="table-box box-dsh">Customer Header<br>name · city · tags</div>
      <div class="table-box box-dsh">Total Balance<br>KPI card</div>
      <div class="table-box box-dsh">Loan Exposure<br>KPI card</div>
      <div class="table-box box-dsh">Credit Score<br>KPI card</div>
      <div class="table-box box-dsh">Monthly Spend<br>KPI card</div>
      <div class="table-box box-dsh">Engagement Score<br>KPI card</div>
      <div class="table-box box-dsh">Churn Risk<br>KPI card</div>
      <div class="table-box box-dsh">Product Holdings<br>bar chart</div>
      <div class="table-box box-dsh">Monthly Spending<br>line chart</div>
      <div class="table-box box-dsh">Channel Usage<br>bar chart</div>
      <div class="table-box box-dsh">Churn Risk Trend<br>line chart</div>
      <div class="table-box box-dsh">Activity Feed<br>table</div>
    </div>
  </div>
  <div class="section-note">
    One customer_id filter controls all 12 widgets simultaneously ·
    Switch any customer to see their complete 360° profile
  </div>

</div>
"""
displayHTML(html_architecture)

---
## Live Layer Health Check
### Bronze Layer — Row Counts

In [0]:
%sql
SELECT 'bronze_customers'             AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_customers              UNION ALL
SELECT 'bronze_accounts'              AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_accounts               UNION ALL
SELECT 'bronze_cards'                 AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_cards                  UNION ALL
SELECT 'bronze_loans'                 AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_loans                  UNION ALL
SELECT 'bronze_branches'              AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_branches               UNION ALL
SELECT 'bronze_merchants'             AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_merchants              UNION ALL
SELECT 'bronze_transactions'          AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_transactions           UNION ALL
SELECT 'bronze_digital_activity'      AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_digital_activity       UNION ALL
SELECT 'bronze_loan_payment'          AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_loan_payment           UNION ALL
SELECT 'bronze_customer_interactions' AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_customer_interactions  UNION ALL
SELECT 'bronze_notifications'         AS table_name, COUNT(*) AS rows, 'Bronze' AS layer FROM customer_360.bronze.bronze_notifications
ORDER BY rows DESC;

### Silver Layer — Row Counts

In [0]:
%sql
SELECT 'silver_customers'              AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_customers              UNION ALL
SELECT 'silver_accounts'               AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_accounts               UNION ALL
SELECT 'silver_cards'                  AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_cards                  UNION ALL
SELECT 'silver_loans'                  AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_loans                  UNION ALL
SELECT 'silver_branches'               AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_branches               UNION ALL
SELECT 'silver_merchants'              AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_merchants              UNION ALL
SELECT 'silver_transactions'           AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_transactions           UNION ALL
SELECT 'silver_digital_activity'       AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_digital_activity       UNION ALL
SELECT 'silver_loan_payments'          AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_loan_payments          UNION ALL
SELECT 'silver_customer_interactions'  AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_customer_interactions  UNION ALL
SELECT 'silver_notifications'          AS table_name, COUNT(*) AS rows, 'Silver' AS layer FROM customer_360.silver.silver_notifications
ORDER BY rows DESC;

### Gold Layer — Row Counts

In [0]:
%sql
SELECT 'customer_360'       AS table_name, COUNT(*) AS rows, 'Gold' AS layer FROM customer_360.gold.customer_360       UNION ALL
SELECT 'monthly_spending'   AS table_name, COUNT(*) AS rows, 'Gold' AS layer FROM customer_360.gold.monthly_spending   UNION ALL
SELECT 'product_holdings'   AS table_name, COUNT(*) AS rows, 'Gold' AS layer FROM customer_360.gold.product_holdings   UNION ALL
SELECT 'channel_usage'      AS table_name, COUNT(*) AS rows, 'Gold' AS layer FROM customer_360.gold.channel_usage      UNION ALL
SELECT 'churn_risk_monthly' AS table_name, COUNT(*) AS rows, 'Gold' AS layer FROM customer_360.gold.churn_risk_monthly UNION ALL
SELECT 'activity_feed'      AS table_name, COUNT(*) AS rows, 'Gold' AS layer FROM customer_360.gold.activity_feed
ORDER BY rows DESC;

### Full Pipeline Summary

In [0]:
%sql
SELECT
  layer,
  COUNT(DISTINCT table_name)  AS total_tables,
  SUM(rows)                   AS total_rows
FROM (
  SELECT 'Bronze' AS layer, 'bronze_customers'             AS table_name, COUNT(*) AS rows FROM customer_360.bronze.bronze_customers             UNION ALL
  SELECT 'Bronze', 'bronze_accounts',              COUNT(*) FROM customer_360.bronze.bronze_accounts              UNION ALL
  SELECT 'Bronze', 'bronze_cards',                 COUNT(*) FROM customer_360.bronze.bronze_cards                 UNION ALL
  SELECT 'Bronze', 'bronze_loans',                 COUNT(*) FROM customer_360.bronze.bronze_loans                 UNION ALL
  SELECT 'Bronze', 'bronze_branches',              COUNT(*) FROM customer_360.bronze.bronze_branches              UNION ALL
  SELECT 'Bronze', 'bronze_merchants',             COUNT(*) FROM customer_360.bronze.bronze_merchants             UNION ALL
  SELECT 'Bronze', 'bronze_transactions',          COUNT(*) FROM customer_360.bronze.bronze_transactions          UNION ALL
  SELECT 'Bronze', 'bronze_digital_activity',      COUNT(*) FROM customer_360.bronze.bronze_digital_activity      UNION ALL
  SELECT 'Bronze', 'bronze_loan_payment',          COUNT(*) FROM customer_360.bronze.bronze_loan_payment          UNION ALL
  SELECT 'Bronze', 'bronze_customer_interactions', COUNT(*) FROM customer_360.bronze.bronze_customer_interactions UNION ALL
  SELECT 'Bronze', 'bronze_notifications',         COUNT(*) FROM customer_360.bronze.bronze_notifications         UNION ALL
  SELECT 'Silver', 'silver_customers',             COUNT(*) FROM customer_360.silver.silver_customers             UNION ALL
  SELECT 'Silver', 'silver_accounts',              COUNT(*) FROM customer_360.silver.silver_accounts              UNION ALL
  SELECT 'Silver', 'silver_cards',                 COUNT(*) FROM customer_360.silver.silver_cards                 UNION ALL
  SELECT 'Silver', 'silver_loans',                 COUNT(*) FROM customer_360.silver.silver_loans                 UNION ALL
  SELECT 'Silver', 'silver_branches',              COUNT(*) FROM customer_360.silver.silver_branches              UNION ALL
  SELECT 'Silver', 'silver_merchants',             COUNT(*) FROM customer_360.silver.silver_merchants             UNION ALL
  SELECT 'Silver', 'silver_transactions',          COUNT(*) FROM customer_360.silver.silver_transactions          UNION ALL
  SELECT 'Silver', 'silver_digital_activity',      COUNT(*) FROM customer_360.silver.silver_digital_activity      UNION ALL
  SELECT 'Silver', 'silver_loan_payments',         COUNT(*) FROM customer_360.silver.silver_loan_payments         UNION ALL
  SELECT 'Silver', 'silver_customer_interactions', COUNT(*) FROM customer_360.silver.silver_customer_interactions UNION ALL
  SELECT 'Silver', 'silver_notifications',         COUNT(*) FROM customer_360.silver.silver_notifications         UNION ALL
  SELECT 'Gold',   'customer_360',                 COUNT(*) FROM customer_360.gold.customer_360                   UNION ALL
  SELECT 'Gold',   'monthly_spending',             COUNT(*) FROM customer_360.gold.monthly_spending               UNION ALL
  SELECT 'Gold',   'product_holdings',             COUNT(*) FROM customer_360.gold.product_holdings               UNION ALL
  SELECT 'Gold',   'channel_usage',                COUNT(*) FROM customer_360.gold.channel_usage                  UNION ALL
  SELECT 'Gold',   'churn_risk_monthly',           COUNT(*) FROM customer_360.gold.churn_risk_monthly             UNION ALL
  SELECT 'Gold',   'activity_feed',                COUNT(*) FROM customer_360.gold.activity_feed
)
GROUP BY layer
ORDER BY CASE layer WHEN 'Bronze' THEN 1
                    WHEN 'Silver' THEN 2
                    ELSE               3 END;